<a href="https://colab.research.google.com/github/tecepeipe/ollama-colab-runner/blob/main/ollama_colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ollama Colab Runner**
# <img src='https://ollama.com/public/ollama.png' alt="Ollama"/>
When running this, ideally, select an instance with GPU:<br>
T4 for free ones, A100/L4 for paid subscribers<br><br>
Run each of the 3 cells, before running your prompt.<br>
If you interrupt execution, start the server again

In [ ]:
# @title Install components
!apt-get install -y pciutils lshw
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers

import os
# Set LD_LIBRARY_PATH so the system NVIDIA library
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

In [47]:
# @title Start server and API endpoint
import subprocess
import os

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"
env["OLLAMA_ORIGINS"] = "*"
process = subprocess.Popen(['ollama', 'serve'], env=env)

In [ ]:
# @title Select your model
model = "gemma4:e4b" # @param ["gemma4:e4b","deepseek-r1:7b","deepseek-r1:14b","deepseek-r1:32b","deepseek-coder:1.3b","deepseek-coder:6.7b","deepseek-coder:33b","mistral:7b","phi4:14b","qwen3.5:9b","qwen3.6:27b"]
!ollama pull {model}

In [ ]:
# @title Interacting with the model (streaming)
Prompt = "Why is the sky blue?"  # @param {"type":"string"}
Think = True # @param {"type":"boolean"}
Show_Thinking = False # @param {"type":"boolean"}

from IPython.display import display, Markdown
import ollama

# 1. Prepare the message payload based on the "Think" checkbox
if Think:
    messages = [
        {
            "role": "system",
            "content": "<|think|> You are a precise reasoning assistant."
        },
        {
            "role": "user",
            "content": Prompt
        }
    ]
else:
    messages = [
        {
            "role": "user",
            "content": Prompt
        }
    ]

# 2. Start streaming response
stream = ollama.chat(
    model=model,
    messages=messages,
    stream=True,
    think=Think # Tells Ollama to explicitly catch the thinking block if the model uses it
)

# 3. Collect and display streamed chunks safely
full_response = ""
started_thinking = False

for chunk in stream:
    # Handle the reasoning track (if the model separates it into its own field)
    if 'thinking' in chunk['message'] and chunk['message']['thinking']:
        if Show_Thinking:  # Only print to terminal if the user wants to see it
            if not started_thinking:
                print("--- Thinking Process ---")
                started_thinking = True
            thought = chunk['message']['thinking']
            print(thought, end="", flush=True)

    # Handle the standard text content response
    elif 'content' in chunk['message'] and chunk['message']['content']:
        if started_thinking and Show_Thinking:
            print("\n\n--- Final Response ---")
            started_thinking = False # Reset flag so we only print the divider once

        content = chunk['message']['content']
        full_response += content
        print(content, end="", flush=True)

print("\n") # Add a final newline for neatness

# Optional Markdown rendering after completion
display(Markdown(full_response))

In [ ]:
# @title Exposing Ollama REST API to use it as local LLM or VS Code
!npm install -g localtunnel # Cloudflare free Tunnels
# Start a background tunnel pointing to Ollama's default port
!lt --port 11434

In [ ]:
# @title If changing models, cancel the tunnel and execute this to kill Ollama
!pkill ollama